# 09 — Análise de Personas nas entrevistas Kyra

Esta versão mantém a lógica da v4 com nomes manuais de personas, mas refaz as principais tabelas e visualizações em **Plotly**, para deixar a leitura mais clara, interativa e pronta para apresentação.

Melhorias visuais desta versão:

1. **Heatmap interativo e legível**: labels quebrados, nomes manuais das personas e valores de lift anotados.
2. **Tabelas interativas em Plotly**: principais tabelas aparecem como `go.Table` e também continuam salvas em CSV.
3. **Gráficos HTML exportados**: os gráficos são salvos em `outputs/personas/figures/*.html`; quando `kaleido` estiver instalado, também tenta salvar PNG.
4. **Nomes manuais na solução macro**: `Cuidadora Consciente`, `Comparadora Omnicanal` e `Compradora Social e Presenteadora`.

A lógica principal continua evitando usar metadados como `projeto`, `país`, `marca` e `público` dentro da clusterização. Esses campos entram só para diagnóstico e leitura posterior, reduzindo o risco de a persona virar apenas um recorte de base.

## 1. Dependências e configuração

O notebook usa `pandas`, `numpy`, `sklearn` e `plotly`. Ele tenta instalar o que estiver faltando. Se o ambiente corporativo bloquear instalação automática, rode a instalação no terminal e volte para esta célula.

In [1]:
import sys
import importlib.util
import subprocess

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "plotly": "plotly",
}

missing = []
for module_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        missing.append(package_name)

if missing:
    print("Instalando pacotes faltantes:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Dependências principais OK.")

Dependências principais OK.


In [2]:
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
import json
import math
import os
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.compose import ColumnTransformer

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 240)

RANDOM_STATE = 42
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print("RUN_ID:", RUN_ID)

# Renderer para notebooks. Se seu ambiente não mostrar o gráfico, troque para: pio.renderers.default = "notebook_connected"
pio.renderers.default = "plotly_mimetype+notebook"


RUN_ID: 20260608_072530


## 2. Paths do projeto

Este bloco procura a raiz do projeto automaticamente. O notebook foi feito para ficar dentro da pasta `notebooks/`, junto com os outros notebooks.

In [3]:
def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for c in candidates:
        if (c / "data" / "processed").exists() and (c / "notebooks").exists():
            return c
    # fallback: se estiver rodando direto dentro de notebooks
    if start.name == "notebooks" and (start.parent / "data" / "processed").exists():
        return start.parent
    raise FileNotFoundError(
        "Não encontrei a raiz do projeto. Rode este notebook dentro da pasta do projeto Kyra, "
        "de preferência em kyrav2/notebooks/."
    )

PROJECT_ROOT = find_project_root()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "personas"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
REPORT_DIR = OUTPUT_DIR / "reports"

for p in [OUTPUT_DIR, TABLE_DIR, FIG_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_ROOT: /Users/emanuelgandra/Desktop/Faculdade/6Período/Kyra/kyrapesquisa/Kyra Atual
OUTPUT_DIR: /Users/emanuelgandra/Desktop/Faculdade/6Período/Kyra/kyrapesquisa/Kyra Atual/outputs/personas


## 3. Carregamento das bases

Preferimos CSV para evitar dependência de `pyarrow`, mas o notebook também tenta ler Parquet quando necessário. A unidade principal da análise será `doc_id`, isto é, uma entrevista/sessão.

In [4]:
def read_first_available(paths, **kwargs) -> pd.DataFrame:
    last_error = None
    for path in paths:
        path = Path(path)
        if not path.exists():
            continue
        try:
            if path.suffix.lower() == ".csv":
                print(f"Lendo CSV: {path.relative_to(PROJECT_ROOT)}")
                return pd.read_csv(path, **kwargs)
            if path.suffix.lower() == ".parquet":
                print(f"Lendo Parquet: {path.relative_to(PROJECT_ROOT)}")
                return pd.read_parquet(path, **kwargs)
        except Exception as e:
            last_error = e
            print(f"Falhou ao ler {path.name}: {e}")
    raise FileNotFoundError(f"Nenhum arquivo disponível pôde ser lido. Último erro: {last_error}")

DOCS_PATHS = [
    DATA_PROCESSED / "interviews_docs_preprocessed.csv",
    DATA_PROCESSED / "interviews_docs_preprocessed.parquet",
]
CHUNKS_PATHS = [
    PROJECT_ROOT / "outputs" / "clusterizacao" / "chunks_with_clusters_topics.csv",
    PROJECT_ROOT / "outputs" / "clusterizacao" / "chunks_with_clusters.csv",
    DATA_PROCESSED / "interviews_chunks_modeling.csv",
    DATA_PROCESSED / "interviews_chunks_modeling.parquet",
]
TURNS_PATHS = [
    DATA_PROCESSED / "interviews_turns_preprocessed.csv",
    DATA_PROCESSED / "interviews_turns_preprocessed.parquet",
]

docs = read_first_available(DOCS_PATHS)
chunks = read_first_available(CHUNKS_PATHS)
turns = read_first_available(TURNS_PATHS)

print("docs:", docs.shape)
print("chunks:", chunks.shape)
print("turns:", turns.shape)

Lendo CSV: data/processed/interviews_docs_preprocessed.csv
Lendo CSV: outputs/clusterizacao/chunks_with_clusters_topics.csv
Lendo CSV: data/processed/interviews_turns_preprocessed.csv
docs: (155, 45)
chunks: (4976, 83)
turns: (69459, 15)


In [5]:
print("Colunas docs:")
plotly_table(pd.DataFrame({"coluna": docs.columns}), "Colunas disponíveis em docs", "tabela_colunas_docs", max_rows=120)
print("Colunas chunks:")
plotly_table(pd.DataFrame({"coluna": chunks.columns}), "Colunas disponíveis em chunks", "tabela_colunas_chunks", max_rows=160)
print("Colunas turns:")
plotly_table(pd.DataFrame({"coluna": turns.columns}), "Colunas disponíveis em turns", "tabela_colunas_turns", max_rows=160)

Colunas docs:


NameError: name 'plotly_table' is not defined

## 4. Funções utilitárias de texto

Estas funções padronizam texto, extraem evidências e criam resumos legíveis. Elas não removem informação demais: para personas, nuances importam.

In [ ]:
def strip_accents(text: str) -> str:
    text = str(text)
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))

def norm_text(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = strip_accents(text.lower())
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[\w\.-]+@[\w\.-]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def short_excerpt(text: str, max_chars: int = 420) -> str:
    text = re.sub(r"\s+", " ", "" if pd.isna(text) else str(text)).strip()
    if len(text) <= max_chars:
        return text
    cut = text[:max_chars].rsplit(" ", 1)[0]
    return cut + "..."

def safe_join(values, sep=" | "):
    vals = [str(v) for v in values if pd.notna(v) and str(v).strip()]
    return sep.join(vals)

def save_csv(df: pd.DataFrame, filename: str, index: bool = False) -> Path:
    path = TABLE_DIR / filename
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print(f"Salvo: {path.relative_to(PROJECT_ROOT)} shape={df.shape}")
    return path



def save_plotly_fig(fig, filename_stem: str, width: int | None = None, height: int | None = None):
    """Mostra e salva uma figura Plotly em HTML; tenta salvar PNG se kaleido estiver disponível."""
    if width or height:
        fig.update_layout(width=width, height=height)
    html_path = FIG_DIR / f"{filename_stem}.html"
    fig.write_html(html_path, include_plotlyjs="cdn")
    print(f"HTML salvo: {html_path.relative_to(PROJECT_ROOT)}")
    try:
        png_path = FIG_DIR / f"{filename_stem}.png"
        fig.write_image(png_path, scale=2)
        print(f"PNG salvo: {png_path.relative_to(PROJECT_ROOT)}")
    except Exception as e:
        print("PNG não salvo. Para exportar PNG estático, instale kaleido: pip install kaleido")
    fig.show()
    return html_path


def wrap_label(text, width=22):
    """Quebra labels longos para eixos/heatmaps em HTML."""
    text = str(text)
    words = text.split()
    lines, current = [], ""
    for w in words:
        if len(current) + len(w) + 1 <= width:
            current = (current + " " + w).strip()
        else:
            if current:
                lines.append(current)
            current = w
    if current:
        lines.append(current)
    return "<br>".join(lines)


def plotly_table(df: pd.DataFrame, title: str = "Tabela", filename_stem: str | None = None, max_rows: int = 50, height: int | None = None):
    """Exibe tabela interativa em Plotly e salva em HTML quando filename_stem for informado."""
    show_df = df.head(max_rows).copy()
    # Formatação leve para percentuais e floats
    for col in show_df.columns:
        if pd.api.types.is_float_dtype(show_df[col]):
            show_df[col] = show_df[col].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")
    values = [show_df[col].astype(str).tolist() for col in show_df.columns]
    fig = go.Figure(data=[go.Table(
        header=dict(values=[f"<b>{c}</b>" for c in show_df.columns], align="left", height=30),
        cells=dict(values=values, align="left", height=26)
    )])
    fig.update_layout(
        title=title,
        height=height or min(900, 160 + 28 * max(1, len(show_df))),
        margin=dict(l=10, r=10, t=55, b=10)
    )
    if filename_stem:
        html_path = FIG_DIR / f"{filename_stem}.html"
        fig.write_html(html_path, include_plotlyjs="cdn")
        print(f"Tabela HTML salva: {html_path.relative_to(PROJECT_ROOT)}")
    fig.show()
    return show_df


def save_md(text: str, filename: str) -> Path:
    path = REPORT_DIR / filename
    path.write_text(text, encoding="utf-8")
    print(f"Salvo: {path.relative_to(PROJECT_ROOT)}")
    return path

## 5. Construção de texto por entrevista

A análise de personas precisa estar em nível de entrevista. Para reduzir o peso do entrevistador, o notebook tenta montar um texto dos participantes a partir de `turns`. Quando a identificação não é clara, ele usa o texto limpo do documento como fallback.

Regra prática usada aqui:

- dentro de cada `doc_id`, o speaker com maior taxa de perguntas tende a ser entrevistador;
- os demais speakers entram como fala do participante;
- se sobrar texto muito curto, usamos `texto_clean` do documento.

In [ ]:
# Colunas de texto disponíveis
DOC_TEXT_COL = "texto_clean" if "texto_clean" in docs.columns else None
CHUNK_TEXT_COL = "text_for_embedding" if "text_for_embedding" in chunks.columns else "texto_preprocessado"
TURN_TEXT_COL = "turn_text_clean" if "turn_text_clean" in turns.columns else "turn_text_raw"

required = ["doc_id"]
for df_name, df in [("docs", docs), ("chunks", chunks), ("turns", turns)]:
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{df_name} sem colunas obrigatórias: {missing_cols}")

# Limpeza básica
for df in [docs, chunks, turns]:
    df["doc_id"] = df["doc_id"].astype(str)

turns["turn_text_norm"] = turns[TURN_TEXT_COL].map(norm_text)
turns["is_question_turn"] = turns.get("is_question_turn", False).fillna(False).astype(bool)

# Identificar provável entrevistador por doc: mais perguntas e participação relevante
speaker_stats = (
    turns.groupby(["doc_id", "speaker_id"], dropna=False)
    .agg(
        n_turns=("turn_index", "count"),
        n_words=("n_words_turn", "sum"),
        question_rate=("is_question_turn", "mean"),
    )
    .reset_index()
)

speaker_stats["interviewer_score"] = (
    speaker_stats["question_rate"].fillna(0) * 0.75
    + (speaker_stats["n_turns"] / speaker_stats.groupby("doc_id")["n_turns"].transform("sum")).fillna(0) * 0.25
)

likely_interviewer = (
    speaker_stats.sort_values(["doc_id", "interviewer_score"], ascending=[True, False])
    .groupby("doc_id")
    .head(1)[["doc_id", "speaker_id"]]
    .rename(columns={"speaker_id": "likely_interviewer_speaker_id"})
)

turns2 = turns.merge(likely_interviewer, on="doc_id", how="left")
turns2["is_likely_interviewer"] = turns2["speaker_id"].astype(str) == turns2["likely_interviewer_speaker_id"].astype(str)

participant_turns = turns2.loc[~turns2["is_likely_interviewer"]].copy()
participant_text_by_doc = (
    participant_turns.groupby("doc_id")["turn_text_clean" if "turn_text_clean" in participant_turns.columns else TURN_TEXT_COL]
    .apply(lambda s: "\n".join([str(x) for x in s.dropna().tolist()]))
    .rename("participant_text_from_turns")
    .reset_index()
)

# Texto agregado por chunks como fallback adicional
chunks["chunk_text_norm"] = chunks[CHUNK_TEXT_COL].map(norm_text)
chunk_text_by_doc = (
    chunks.groupby("doc_id")[CHUNK_TEXT_COL]
    .apply(lambda s: "\n".join([str(x) for x in s.dropna().tolist()]))
    .rename("text_from_chunks")
    .reset_index()
)

base_cols = [c for c in [
    "doc_id", "projeto", "pais", "marca_foco", "publico", "tipo_sessao", "canal_origem", "contexto_origem",
    "mes", "mes_num", "ano", "idioma_original", "duracao_aprox_min", "n_words_clean", "qualidade_texto_global"
] if c in docs.columns]

doc_base = docs[base_cols].drop_duplicates("doc_id").copy()
if DOC_TEXT_COL:
    doc_base = doc_base.merge(docs[["doc_id", DOC_TEXT_COL]].drop_duplicates("doc_id"), on="doc_id", how="left")
else:
    doc_base["texto_clean"] = ""
    DOC_TEXT_COL = "texto_clean"

doc_base = doc_base.merge(participant_text_by_doc, on="doc_id", how="left")
doc_base = doc_base.merge(chunk_text_by_doc, on="doc_id", how="left")

def choose_persona_text(row):
    pt = row.get("participant_text_from_turns", "")
    dt = row.get(DOC_TEXT_COL, "")
    ct = row.get("text_from_chunks", "")
    # Usa fala de participante se tiver volume mínimo; caso contrário usa texto limpo do doc; depois chunks.
    if isinstance(pt, str) and len(pt.split()) >= 120:
        return pt
    if isinstance(dt, str) and len(dt.split()) >= 120:
        return dt
    return ct if isinstance(ct, str) else ""

doc_base["persona_text"] = doc_base.apply(choose_persona_text, axis=1)
doc_base["persona_text_norm"] = doc_base["persona_text"].map(norm_text)
doc_base["n_words_persona_text"] = doc_base["persona_text_norm"].str.split().map(len)

plotly_table(doc_base[["doc_id", "projeto", "pais", "marca_foco", "publico", "n_words_persona_text"]].head())
print("Entrevistas com texto suficiente:", (doc_base["n_words_persona_text"] >= 120).sum(), "/", len(doc_base))

## 6. Dimensões interpretáveis de decisão

Aqui criamos features por dicionário. A lista abaixo é propositalmente editável: depois de revisar os resultados, você deve ajustar palavras, adicionar dimensões ou remover termos que estejam gerando ruído.

A diferença principal em relação a tópicos automáticos é que cada dimensão responde a uma pergunta de negócio, por exemplo: “essa persona decide por preço?”, “valoriza naturalidade/sustentabilidade?”, “fala de eficácia?”, “é orientada a presente/status?”, etc.

In [ ]:
DIMENSION_PATTERNS = {
    "preco_valor_promocao": [
        "preco", "caro", "barato", "valor", "custo", "promocao", "oferta", "desconto", "cupom", "econom",
        "vale a pena", "pagar", "precios", "precio", "costoso", "oferta", "descuento", "promocion"
    ],
    "eficacia_resultado_funcional": [
        "funciona", "resultado", "eficaz", "eficien", "resolve", "melhora", "tratamento", "beneficio", "qualidade",
        "durabilidade", "fixacao", "hidrata", "limpa", "protege", "efectivo", "resultado", "beneficio"
    ],
    "sensorial_perfume_prazer": [
        "cheiro", "perfume", "fragrancia", "aroma", "gostoso", "textura", "sensacao", "macio", "suave",
        "prazer", "relax", "olor", "fragancia", "suavidad", "sensacion"
    ],
    "natural_sustentavel_consciente": [
        "natural", "natureza", "sustent", "refil", "recicl", "plastico", "ecologico", "vegano", "cruelty",
        "organico", "amazonia", "ingrediente", "medio ambiente", "sostenible", "reciclable"
    ],
    "marca_confianca_tradicao": [
        "marca", "confio", "confianca", "tradicao", "conhecida", "reputacao", "seguranca", "natura", "boticario",
        "avon", "recomenda", "referencia", "confianza", "tradicional"
    ],
    "status_presente_estetica": [
        "presente", "presentear", "bonito", "premium", "sofistic", "chique", "luxo", "elegante", "caixa",
        "kit", "visual", "design", "regalo", "regalar", "lujo", "bonita"
    ],
    "praticidade_rotina_conveniencia": [
        "pratico", "facil", "rapido", "rotina", "dia a dia", "correria", "tempo", "simples", "convenien",
        "aplicar", "usar", "dificil", "complicado", "facilito", "rapido", "rutina"
    ],
    "canal_loja_consultora_venda": [
        "loja", "consultora", "revista", "pedido", "vendedora", "comprar", "venda", "catalogo", "online",
        "site", "app", "whatsapp", "tienda", "consultora", "catalogo", "pedido"
    ],
    "pele_corpo_cuidado_pessoal": [
        "pele", "rosto", "corpo", "cabelo", "banho", "skincare", "acne", "oleosa", "ressec", "hidrata",
        "cuidado", "piel", "cara", "cabello", "cuerpo"
    ],
    "familia_social_influencia": [
        "familia", "mae", "filha", "amiga", "amigo", "marido", "esposa", "indicacao", "recomenda", "influencer",
        "rede social", "instagram", "tiktok", "familia", "amiga", "recomendacion"
    ],
    "diversidade_identidade_representatividade": [
        "negra", "preta", "morena", "cabelo cacheado", "represent", "inclus", "diversidade", "identidade",
        "autoestima", "rac", "diversidad", "representacion", "inclusion"
    ],
    "inovacao_tecnologia_novidade": [
        "novo", "novidade", "inov", "tecnologia", "diferente", "moderno", "lancamento", "tendencia",
        "nuevo", "novedad", "innovacion", "tecnologia", "moderno"
    ],
}

# Sinais de atitude. São úteis para diferenciar perfis mesmo dentro do mesmo tema.
ATTITUDE_PATTERNS = {
    "positivo_encantamento": ["amei", "adoro", "gosto muito", "maravilhoso", "perfeito", "lindo", "encanta", "apaixon", "me encanta", "me gusta mucho"],
    "negativo_rejeicao": ["nao gosto", "odeio", "ruim", "horrivel", "decepcion", "problema", "irrita", "nao compraria", "no me gusta", "malo"],
    "duvida_ceticismo": ["nao sei", "talvez", "depende", "duvida", "estranho", "confuso", "parece que", "no se", "depende"],
    "comparacao_concorrencia": ["melhor que", "pior que", "compar", "versus", "concorrente", "outra marca", "boticario", "natura", "avon", "eudora"],
}

ALL_PATTERNS = {**DIMENSION_PATTERNS, **ATTITUDE_PATTERNS}

compiled_patterns = {
    dim: re.compile(r"(" + "|".join(re.escape(norm_text(term)) for term in terms) + r")", re.IGNORECASE)
    for dim, terms in ALL_PATTERNS.items()
}

def count_pattern_hits(text_norm: str, pattern: re.Pattern) -> int:
    if not isinstance(text_norm, str) or not text_norm:
        return 0
    return len(pattern.findall(text_norm))

feature_rows = []
for _, row in doc_base.iterrows():
    text = row["persona_text_norm"]
    n_words = max(row["n_words_persona_text"], 1)
    out = {"doc_id": row["doc_id"], "n_words_persona_text": n_words}
    for dim, pat in compiled_patterns.items():
        hits = count_pattern_hits(text, pat)
        out[f"dim__{dim}__hits"] = hits
        out[f"dim__{dim}"] = hits / n_words * 1000
    feature_rows.append(out)

dim_features = pd.DataFrame(feature_rows)
dim_cols = [c for c in dim_features.columns if c.startswith("dim__") and not c.endswith("__hits")]

print("Dimensões criadas:", len(dim_cols))
plotly_table(dim_features[["doc_id", *dim_cols]].head())

## 7. Features auxiliares dos clusters e tópicos existentes

Como a clusterização anterior não funcionou tão bem como resposta final, aqui ela entra apenas como **sinal auxiliar**. Exemplo: se uma entrevista tem muitos chunks associados a embalagem/refil e também alto score de sustentabilidade, isso ajuda a descrever a persona.

In [ ]:
aux_parts = []
label_cols_available = [c for c in ["cluster_label", "cluster_auto_label", "nmf_topic_id", "nmf_topic_label", "lda_topic_id", "lda_topic_label"] if c in chunks.columns]

for col in label_cols_available:
    tmp = (
        chunks.dropna(subset=[col])
        .groupby(["doc_id", col])
        .size()
        .rename("n")
        .reset_index()
    )
    if tmp.empty:
        continue
    pivot = tmp.pivot_table(index="doc_id", columns=col, values="n", fill_value=0)
    pivot = pivot.div(pivot.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    pivot.columns = [f"aux_{col}__{str(c)}" for c in pivot.columns]
    aux_parts.append(pivot.reset_index())

if aux_parts:
    aux_features = aux_parts[0]
    for part in aux_parts[1:]:
        aux_features = aux_features.merge(part, on="doc_id", how="outer")
    aux_features = aux_features.fillna(0)
else:
    aux_features = pd.DataFrame({"doc_id": doc_base["doc_id"]})

print("Features auxiliares de clusters/tópicos:", aux_features.shape)
plotly_table(aux_features.head())

## 8. Matriz de modelagem em nível de entrevista

A matriz combina:

- TF-IDF + SVD do texto da entrevista;
- dimensões interpretáveis por dicionário;
- metadados de projeto/público/marca/país como controles;
- sinais auxiliares dos clusters/tópicos existentes.

A clusterização final será feita em entrevistas, não em chunks.

In [ ]:
# Filtra entrevistas com texto mínimo
MIN_WORDS = 120

# Defesa contra execução parcial: garante que a coluna de tamanho exista.
if "n_words_persona_text" not in doc_base.columns:
    if "persona_text_norm" not in doc_base.columns:
        doc_base["persona_text_norm"] = doc_base["persona_text"].map(norm_text) if "persona_text" in doc_base.columns else ""
    doc_base["n_words_persona_text"] = doc_base["persona_text_norm"].fillna("").str.split().map(len)

model_docs = doc_base.loc[doc_base["n_words_persona_text"] >= MIN_WORDS].copy()
print("Entrevistas elegíveis:", model_docs.shape)

# Metadados categóricos: serão usados para PERFIL e diagnóstico, não como input principal da clusterização.
# Isso reduz o risco de criar uma persona que é só "projeto X" ou "país Y".
categorical_cols = [c for c in ["projeto", "pais", "marca_foco", "publico", "tipo_sessao"] if c in model_docs.columns]
for c in categorical_cols:
    model_docs[c] = model_docs[c].fillna("nao_inferido").astype(str).str.slice(0, 80)

# TF-IDF em texto agregado
CUSTOM_STOPWORDS = sorted(set("""
a o os as um uma uns umas de da do das dos e ou que para por com sem em no na nos nas ao aos a gente voce vocês você pra pro porque entao assim tipo ne né ta tá ai aí eu ele ela eles elas meu minha meus minhas seu sua seus suas isso isto aquilo aqui ali lá mais menos muito muita muitos muitas pouco pouca todos todas todo toda ser estar ter fazer falar pensar achar coisa coisas vez vezes sobre tambem também desde como quando onde qual quais quem cada outro outra outros outras nao não sim mas se ja já le la el los las un una unos unas y que para por con sin en del de al lo es son soy fue era muy mas más menos todo toda todos todas esto esta este esa ese eso aqui ahí como cuando donde cual cuales quien pero si no ya
""".split()))

# Limita max_features para rodar em notebook comum
MAX_TFIDF_FEATURES = 3500
N_SVD_COMPONENTS = 40

vectorizer = TfidfVectorizer(
    min_df=2,
    max_df=0.85,
    max_features=MAX_TFIDF_FEATURES,
    ngram_range=(1, 2),
    stop_words=CUSTOM_STOPWORDS,
    strip_accents="unicode",
)

X_tfidf = vectorizer.fit_transform(model_docs["persona_text_norm"].fillna(""))
n_svd = min(N_SVD_COMPONENTS, max(2, X_tfidf.shape[1] - 1), max(2, X_tfidf.shape[0] - 1))
svd = TruncatedSVD(n_components=n_svd, random_state=RANDOM_STATE)
X_text = svd.fit_transform(X_tfidf)
print("X_tfidf:", X_tfidf.shape, "X_text/SVD:", X_text.shape, "variância explicada:", round(svd.explained_variance_ratio_.sum(), 3))

# Features interpretáveis
model_features = model_docs[["doc_id"]].merge(dim_features, on="doc_id", how="left")
model_features = model_features.merge(aux_features, on="doc_id", how="left").fillna(0)

# Separa dimensões interpretáveis de features auxiliares dos clusters/tópicos antigos.
# As dimensões têm peso maior porque explicam comportamento. Os tópicos antigos entram como sinal fraco.
feature_cols = [c for c in model_features.columns if c != "doc_id" and not c.endswith("__hits")]
dim_feature_cols = [c for c in feature_cols if c.startswith("dim__") or c.startswith("att__")]
aux_feature_cols = [c for c in feature_cols if c.startswith("aux_")]

X_dim = model_features[dim_feature_cols].replace([np.inf, -np.inf], 0).fillna(0).to_numpy(dtype=float) if dim_feature_cols else np.empty((len(model_docs), 0))
X_aux = model_features[aux_feature_cols].replace([np.inf, -np.inf], 0).fillna(0).to_numpy(dtype=float) if aux_feature_cols else np.empty((len(model_docs), 0))

# Guarda os scalers para permitir classificar entrevistas novas depois.
text_scaler = StandardScaler(with_mean=True, with_std=True)
X_text_scaled = text_scaler.fit_transform(X_text)

if X_dim.shape[1]:
    dim_scaler = StandardScaler(with_mean=True, with_std=True)
    X_dim_scaled = dim_scaler.fit_transform(X_dim)
else:
    dim_scaler = None
    X_dim_scaled = X_dim

if X_aux.shape[1]:
    aux_scaler = StandardScaler(with_mean=True, with_std=True)
    X_aux_scaled = aux_scaler.fit_transform(X_aux)
else:
    aux_scaler = None
    X_aux_scaled = X_aux

# Peso da v2:
# - texto ainda importa;
# - dimensões interpretáveis passam a ter peso alto;
# - clusters/tópicos antigos ficam leves;
# - metadados ficam fora do modelo e entram só no diagnóstico posterior.
TEXT_WEIGHT = 0.80
DIMENSION_WEIGHT = 1.20
AUX_OLD_CLUSTER_WEIGHT = 0.10

X_model = np.hstack([
    X_text_scaled * TEXT_WEIGHT,
    X_dim_scaled * DIMENSION_WEIGHT,
    X_aux_scaled * AUX_OLD_CLUSTER_WEIGHT,
])
X_model = normalize(X_model)

print("X_model:", X_model.shape)
print("Dimensões interpretáveis:", len(dim_feature_cols), "Auxiliares antigos:", len(aux_feature_cols), "Metadados usados no modelo: 0")


## 9. Escolha do número de personas

Em vez de escolher um `k` fixo, testamos uma faixa e comparamos métricas. A escolha automática prioriza silhouette, mas você deve olhar também se o número de personas é **interpretável para negócio**.

In [ ]:
N = X_model.shape[0]
if N < 6:
    raise ValueError("Poucas entrevistas elegíveis para clusterização. Reduza MIN_WORDS ou revise a base.")

k_min = 3
k_max = min(10, max(3, N // 4))
ks = list(range(k_min, k_max + 1))

# Parâmetros de decisão de negócio.
# O silhouette tende a subir quando k fica alto, mas isso pode criar micro-personas pouco acionáveis.
# Por padrão, a v2 favorece uma solução de 3 a 6 personas para apresentação.
PREFER_BUSINESS_K_UNTIL = 6
MIN_CLUSTER_SHARE_TARGET = 0.07
COMPLEXITY_PENALTY_PER_EXTRA_K = 0.020
SMALL_CLUSTER_PENALTY = 0.35
DOMINANCE_PENALTY = 0.08
STABILITY_BONUS = 0.03

scores = []
stability_seeds = [11, 22, 33, 44, 55]

for k in ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=50)
    labels = km.fit_predict(X_model)
    counts_abs = pd.Series(labels).value_counts()
    counts = counts_abs / counts_abs.sum()

    if len(set(labels)) > 1:
        sil = silhouette_score(X_model, labels)
        ch = calinski_harabasz_score(X_model, labels)
        db = davies_bouldin_score(X_model, labels)
    else:
        sil, ch, db = np.nan, np.nan, np.nan

    # Estabilidade simples: compara a solução base com soluções de outras seeds.
    ari_scores = []
    for seed in stability_seeds:
        alt_labels = KMeans(n_clusters=k, random_state=seed, n_init=20).fit_predict(X_model)
        ari_scores.append(adjusted_rand_score(labels, alt_labels))
    stability = float(np.nanmean(ari_scores)) if ari_scores else np.nan

    largest_share = float(counts.max())
    smallest_share = float(counts.min())
    min_cluster_n = int(counts_abs.min())

    small_cluster_gap = max(0.0, MIN_CLUSTER_SHARE_TARGET - smallest_share)
    dominance_gap = max(0.0, largest_share - 0.40)
    complexity_gap = max(0, k - k_min)

    # Score mais conservador do que silhouette puro.
    selection_score = (
        float(sil)
        - COMPLEXITY_PENALTY_PER_EXTRA_K * complexity_gap
        - SMALL_CLUSTER_PENALTY * small_cluster_gap
        - DOMINANCE_PENALTY * dominance_gap
        + STABILITY_BONUS * stability
    )

    scores.append({
        "k": k,
        "silhouette": sil,
        "calinski_harabasz": ch,
        "davies_bouldin": db,
        "stability_ari_mean": stability,
        "largest_persona_share": largest_share,
        "smallest_persona_share": smallest_share,
        "min_cluster_n": min_cluster_n,
        "selection_score": selection_score,
        "business_candidate": k <= PREFER_BUSINESS_K_UNTIL,
    })

k_eval = pd.DataFrame(scores)

# Melhor solução para apresentação: primeiro procura dentro de k <= 6.
business_pool = k_eval[k_eval["business_candidate"]].copy()
if business_pool.empty:
    business_pool = k_eval.copy()

BUSINESS_K = int(business_pool.sort_values("selection_score", ascending=False).iloc[0]["k"])
SILHOUETTE_K = int(k_eval.sort_values("silhouette", ascending=False).iloc[0]["k"])
BEST_K = BUSINESS_K

k_eval_display = k_eval.sort_values("selection_score", ascending=False)
plotly_table(k_eval_display)
save_csv(k_eval_display, "personas_k_selection.csv")

print("K com maior silhouette puro:", SILHOUETTE_K)
print("BEST_K sugerido para personas acionáveis:", BEST_K)
print("Observação: se a leitura qualitativa ficar ampla demais, teste manualmente FINAL_K =", SILHOUETTE_K)


In [ ]:
plot_df = k_eval.sort_values("k")
plot_df["k_label"] = plot_df["k"].astype(str)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_df["k"], y=plot_df["silhouette"],
    mode="lines+markers", name="Silhouette puro",
    hovertemplate="k=%{x}<br>Silhouette=%{y:.3f}<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=plot_df["k"], y=plot_df["selection_score"],
    mode="lines+markers", name="Score acionável",
    hovertemplate="k=%{x}<br>Score acionável=%{y:.3f}<extra></extra>"
))
fig.add_vline(x=BEST_K, line_dash="dash", annotation_text=f"BEST_K = {BEST_K}", annotation_position="top left")
fig.update_layout(
    title="Seleção de k para personas: qualidade estatística vs acionabilidade",
    xaxis_title="Número de personas (k)",
    yaxis_title="Score",
    hovermode="x unified",
    legend_title_text="Métrica",
    template="plotly_white",
    margin=dict(l=60, r=30, t=80, b=60),
)
save_plotly_fig(fig, "personas_k_selection_v5_plotly", width=1100, height=520)

plotly_table(
    plot_df[["k", "silhouette", "selection_score", "min_cluster_share", "max_cluster_share", "stability_mean", "recommended"]],
    "Tabela de seleção de k",
    "tabela_personas_k_selection",
    max_rows=20,
)

## 10. Clusterização final de personas

A partir daqui, criamos a persona final por entrevista. Se você quiser forçar um número diferente de personas, altere `FINAL_K` abaixo.

In [ ]:
# Por padrão, usa o BEST_K conservador.
# Para comparar soluções, altere manualmente para 4, 5, 6 ou para SILHOUETTE_K.
FINAL_K = BEST_K

persona_model = KMeans(n_clusters=FINAL_K, random_state=RANDOM_STATE, n_init=80)
model_docs["persona_num"] = persona_model.fit_predict(X_model)
model_docs["persona_id"] = model_docs["persona_num"].map(lambda x: f"persona_{int(x):02d}")

persona_counts = model_docs["persona_id"].value_counts().sort_index().rename_axis("persona_id").reset_index(name="n_entrevistas")
persona_counts["share_entrevistas"] = persona_counts["n_entrevistas"] / persona_counts["n_entrevistas"].sum()
plotly_table(persona_counts, "Resumo de tamanho das personas", "tabela_persona_counts", max_rows=20)

## 11. Nomeação das personas

Os nomes automáticos continuam disponíveis como apoio, mas nesta versão as **3 personas macro** recebem nomes manuais mais executivos para apresentação e uso no classificador de novas entrevistas.

Mapeamento manual adotado:

- `persona_00` / `macro_k3_p00`: **Cuidadora Consciente**
- `persona_01` / `macro_k3_p01`: **Comparadora Omnicanal**
- `persona_02` / `macro_k3_p02`: **Compradora Social e Presenteadora**

Atenção: os nomes dependem da numeração do KMeans com o `RANDOM_STATE` atual. Se você mudar radicalmente features, seed ou filtros da base, revise 5 a 10 evidências por persona antes de manter os nomes.


In [ ]:
# Junta features interpretáveis e persona
required_persona_cols = ["doc_id", "persona_id", "persona_num", "persona_text", "persona_text_norm", "n_words_persona_text"]
for col in required_persona_cols:
    if col not in model_docs.columns:
        if col == "n_words_persona_text" and "persona_text_norm" in model_docs.columns:
            model_docs[col] = model_docs["persona_text_norm"].fillna("").str.split().map(len)
        elif col in ["persona_text", "persona_text_norm"]:
            model_docs[col] = ""
        else:
            raise KeyError(f"Coluna obrigatória ausente em model_docs: {col}")

persona_base_cols = [c for c in [*required_persona_cols, *categorical_cols] if c in model_docs.columns]
persona_docs = model_docs[persona_base_cols].copy()
persona_docs = persona_docs.merge(dim_features, on="doc_id", how="left")
persona_docs = persona_docs.merge(aux_features, on="doc_id", how="left").fillna(0)

# Top dimensões por lift vs média global
available_dim_cols = [c for c in dim_cols if c in persona_docs.columns]
dim_mean_global = persona_docs[available_dim_cols].mean().replace(0, np.nan)
rows = []
for pid, g in persona_docs.groupby("persona_id"):
    means = g[available_dim_cols].mean()
    lifts = (means / dim_mean_global).replace([np.inf, -np.inf], np.nan).fillna(0)
    for col in available_dim_cols:
        rows.append({
            "persona_id": pid,
            "dimension": col.replace("dim__", "").replace("att__", ""),
            "mean_per_1000_words": means[col],
            "global_mean_per_1000_words": dim_mean_global[col],
            "lift_vs_global": lifts[col],
        })

dim_by_persona = pd.DataFrame(rows).sort_values(["persona_id", "lift_vs_global"], ascending=[True, False])
save_csv(dim_by_persona, "personas_dimension_lifts.csv")
plotly_table(dim_by_persona.groupby("persona_id").head(6))

# Termos característicos por persona usando CountVectorizer/TF-IDF agregado
persona_text_agg = persona_docs.groupby("persona_id")["persona_text_norm"].apply(lambda s: " ".join(s.dropna())).reset_index()
min_df_terms = 2 if len(persona_text_agg) >= 3 else 1
cv = CountVectorizer(max_features=5000, min_df=min_df_terms, ngram_range=(1, 2), stop_words=CUSTOM_STOPWORDS, strip_accents="unicode")
X_counts = cv.fit_transform(persona_text_agg["persona_text_norm"])
terms = np.array(cv.get_feature_names_out())

# TF-IDF/lift simples por persona agregada
persona_term_counts = pd.DataFrame(X_counts.toarray(), index=persona_text_agg["persona_id"], columns=terms)
global_counts = persona_term_counts.sum(axis=0) + 1
persona_totals = persona_term_counts.sum(axis=1).replace(0, np.nan)
global_freq = global_counts / global_counts.sum()

term_rows = []
for pid in persona_term_counts.index:
    freq = (persona_term_counts.loc[pid] + 1) / (persona_totals.loc[pid] + len(terms))
    lift = freq / global_freq
    score = np.log1p(persona_term_counts.loc[pid]) * np.log1p(lift)
    top = score.sort_values(ascending=False).head(18)
    for term, val in top.items():
        term_rows.append({"persona_id": pid, "term": term, "score": val, "count": persona_term_counts.loc[pid, term], "lift": lift[term]})

terms_by_persona = pd.DataFrame(term_rows)
save_csv(terms_by_persona, "personas_top_terms.csv")
plotly_table(terms_by_persona.groupby("persona_id").head(10))


In [ ]:
DIMENSION_LABELS = {
    "preco_valor_promocao": "sensível a preço e valor",
    "eficacia_resultado_funcional": "orientada a eficácia e resultado",
    "sensorial_perfume_prazer": "sensorial e prazer de uso",
    "natural_sustentavel_consciente": "consciente/natural/sustentável",
    "marca_confianca_tradicao": "guiada por marca e confiança",
    "status_presente_estetica": "estética, presente e status",
    "praticidade_rotina_conveniencia": "prática e orientada à rotina",
    "canal_loja_consultora_venda": "canal, loja e consultora",
    "pele_corpo_cuidado_pessoal": "cuidado pessoal e pele/corpo",
    "familia_social_influencia": "social/familiar e influenciada por indicação",
    "diversidade_identidade_representatividade": "identidade e representatividade",
    "inovacao_tecnologia_novidade": "curiosa por inovação e novidade",
    "positivo_encantamento": "entusiasmada/encantada",
    "negativo_rejeicao": "crítica/rejeição explícita",
    "duvida_ceticismo": "cética ou indecisa",
    "comparacao_concorrencia": "comparadora de marcas",
}


# Nomes manuais para a solução macro k=3, definidos após leitura dos outputs.
# Eles são mais úteis para apresentação do que os nomes heurísticos longos.
MANUAL_PERSONA_NAMES_BASE = {
    "persona_00": "Cuidadora Consciente",
    "persona_01": "Comparadora Omnicanal",
    "persona_02": "Compradora Social e Presenteadora",
}

MANUAL_PERSONA_DESCRIPTIONS_BASE = {
    "persona_00": "Cliente ligada a pele, cuidado, naturalidade, eficácia, sensorialidade, inovação e representatividade.",
    "persona_01": "Cliente que compara marcas e canais, avalia Natura/Boticário/Avon/loja/consultora, e decide por confiança, facilidade e valor.",
    "persona_02": "Cliente que compra em contexto social/familiar, presente, indicação, WhatsApp e consultora, mas também verbaliza críticas e objeções.",
}

def make_persona_name(pid: str) -> str:
    top_dims = dim_by_persona[(dim_by_persona["persona_id"] == pid) & (dim_by_persona["lift_vs_global"] > 1.05)].head(3)
    labels = [DIMENSION_LABELS.get(d.replace("dim__", ""), d.replace("_", " ")) for d in top_dims["dimension"].tolist()]
    if not labels:
        labels = ["perfil amplo"]
    return " + ".join(labels[:2]).capitalize()

persona_name_auto_map = {pid: make_persona_name(pid) for pid in sorted(persona_docs["persona_id"].unique())}
# Se houver nome manual para o ID, ele substitui o automático.
persona_name_map = {pid: MANUAL_PERSONA_NAMES_BASE.get(pid, auto_name) for pid, auto_name in persona_name_auto_map.items()}

persona_docs["persona_nome_auto"] = persona_docs["persona_id"].map(persona_name_auto_map)
persona_docs["persona_nome_manual"] = persona_docs["persona_id"].map(persona_name_map)
persona_docs["persona_nome_final"] = persona_docs["persona_nome_manual"].fillna(persona_docs["persona_nome_auto"])
# Mantém compatibilidade com o restante do notebook: a coluna usada nos outputs passa a ser o nome final.
persona_docs["persona_nome_auto"] = persona_docs["persona_nome_final"]

pd.DataFrame({
    "persona_id": list(persona_name_map.keys()),
    "persona_nome_auto_original": [persona_name_auto_map[k] for k in persona_name_map.keys()],
    "persona_nome_final": list(persona_name_map.values()),
    "descricao_manual": [MANUAL_PERSONA_DESCRIPTIONS_BASE.get(k, "") for k in persona_name_map.keys()],
})


## 12. Evidências citáveis por persona

Selecionamos trechos representativos que tenham alta presença das dimensões dominantes da persona. Isso ajuda a validar se a persona faz sentido e a montar apresentação com voz do cliente.

In [ ]:
# Prepara chunks com persona anexada
chunk_evidence_cols = [c for c in [
    "doc_id", "chunk_id", "chunk_index", CHUNK_TEXT_COL, "projeto", "pais", "marca_foco", "publico", "tipo_sessao",
    "cluster_label", "cluster_auto_label", "nmf_topic_id", "nmf_topic_label", "lda_topic_id", "lda_topic_label"
] if c in chunks.columns]
chunks_ev = chunks[chunk_evidence_cols].merge(persona_docs[["doc_id", "persona_id", "persona_nome_auto"]], on="doc_id", how="inner")
chunks_ev["text_norm"] = chunks_ev[CHUNK_TEXT_COL].map(norm_text)
chunks_ev["n_words"] = chunks_ev["text_norm"].str.split().map(len)

# Top dimensões por persona para score de evidência
persona_top_dims = (
    dim_by_persona.sort_values(["persona_id", "lift_vs_global"], ascending=[True, False])
    .groupby("persona_id")
    .head(4)
    .groupby("persona_id")["dimension"]
    .apply(list)
    .to_dict()
)

def evidence_score(row):
    dims = persona_top_dims.get(row["persona_id"], [])
    score = 0
    for dim in dims:
        key = dim.replace("dim__", "")
        pat = compiled_patterns.get(key)
        if pat:
            score += count_pattern_hits(row["text_norm"], pat)
    # Prefere chunks informativos, mas não gigantes
    length_bonus = min(row.get("n_words", 0), 120) / 120
    return score + 0.35 * length_bonus

chunks_ev["evidence_score"] = chunks_ev.apply(evidence_score, axis=1)
chunks_ev = chunks_ev[chunks_ev["n_words"].between(25, 220, inclusive="both")].copy()

ex_rows = []
for pid, g in chunks_ev.sort_values("evidence_score", ascending=False).groupby("persona_id"):
    # Evita pegar 5 trechos do mesmo documento
    used_docs = set()
    selected = []
    for _, r in g.iterrows():
        if r["doc_id"] in used_docs and len(used_docs) < min(5, g["doc_id"].nunique()):
            continue
        selected.append(r)
        used_docs.add(r["doc_id"])
        if len(selected) >= 8:
            break
    for rank, r in enumerate(selected, 1):
        ex_rows.append({
            "persona_id": pid,
            "persona_nome_auto": r["persona_nome_auto"],
            "rank": rank,
            "doc_id": r["doc_id"],
            "chunk_id": r.get("chunk_id", None),
            "evidence_score": r["evidence_score"],
            "excerpt": short_excerpt(r[CHUNK_TEXT_COL], 520),
            "projeto": r.get("projeto", None),
            "pais": r.get("pais", None),
            "marca_foco": r.get("marca_foco", None),
            "publico": r.get("publico", None),
            "tipo_sessao": r.get("tipo_sessao", None),
            "cluster_auto_label": r.get("cluster_auto_label", None),
            "nmf_topic_label": r.get("nmf_topic_label", None),
            "lda_topic_label": r.get("lda_topic_label", None),
        })

evidence = pd.DataFrame(ex_rows)
save_csv(evidence, "personas_evidence_quotes.csv")
plotly_table(evidence.head(20))

## 13. Resumos por persona e cruzamentos

Estas tabelas são as mais úteis para apresentação e discussão com o time: tamanho da persona, dimensões dominantes, segmentos onde ela aparece mais e exemplos de fala.

In [ ]:
summary_rows = []
for pid, g in persona_docs.groupby("persona_id"):
    top_dims = dim_by_persona[dim_by_persona["persona_id"] == pid].head(6)
    top_terms = terms_by_persona[terms_by_persona["persona_id"] == pid].head(12)
    ev = evidence[evidence["persona_id"] == pid].head(3) if "evidence" in globals() else pd.DataFrame()
    row = {
        "persona_id": pid,
        "persona_nome_auto": persona_name_map.get(pid, pid),
        "n_entrevistas": len(g),
        "share_entrevistas": len(g) / len(persona_docs),
        "top_dimensoes": "; ".join([
            f"{DIMENSION_LABELS.get(str(d).replace('dim__','').replace('att__',''), str(d))} (lift {lift:.1f}x)"
            for d, lift in zip(top_dims["dimension"], top_dims["lift_vs_global"])
        ]),
        "top_termos": ", ".join(top_terms["term"].astype(str).tolist()),
        "evidencias_curtas": " || ".join(ev["excerpt"].astype(str).tolist()) if not ev.empty and "excerpt" in ev.columns else "",
    }
    for c in categorical_cols:
        if c in g.columns:
            vc = g[c].fillna("nao_inferido").astype(str).value_counts().head(3)
            row[f"top_{c}"] = "; ".join([f"{idx}: {val}" for idx, val in vc.items()])
            row[f"{c}_top_share"] = float(vc.iloc[0] / len(g)) if len(vc) else np.nan
    summary_rows.append(row)

persona_summary = pd.DataFrame(summary_rows).sort_values("n_entrevistas", ascending=False)
save_csv(persona_summary, "personas_summary.csv")
plotly_table(persona_summary)

# Base final por documento — robusta contra colunas ausentes.
if "persona_nome_auto" not in persona_docs.columns:
    persona_docs["persona_nome_auto"] = persona_docs["persona_id"].map(persona_name_map).fillna(persona_docs["persona_id"])
if "n_words_persona_text" not in persona_docs.columns:
    persona_docs["n_words_persona_text"] = persona_docs.get("persona_text_norm", "").fillna("").str.split().map(len)

final_doc_cols = ["doc_id", "persona_id", "persona_nome_auto", "n_words_persona_text", *categorical_cols]
final_doc_cols = [c for c in final_doc_cols if c in persona_docs.columns]
personas_doc_level = persona_docs[final_doc_cols].copy()
save_csv(personas_doc_level, "personas_doc_level.csv")
plotly_table(personas_doc_level.head())


In [ ]:
# Cruzamentos por segmentação
cross_tables = []
for c in categorical_cols:
    ct = (
        personas_doc_level.groupby([c, "persona_id", "persona_nome_auto"])
        .size()
        .rename("n_entrevistas")
        .reset_index()
    )
    total_seg = ct.groupby(c)["n_entrevistas"].transform("sum")
    total_persona = personas_doc_level["persona_id"].value_counts(normalize=True).to_dict()
    ct["share_no_segmento"] = ct["n_entrevistas"] / total_seg
    ct["share_global_persona"] = ct["persona_id"].map(total_persona)
    ct["lift_vs_global"] = ct["share_no_segmento"] / ct["share_global_persona"].replace(0, np.nan)
    ct.insert(0, "segment_col", c)
    ct = ct.rename(columns={c: "segment_value"})
    cross_tables.append(ct)

personas_by_segment = pd.concat(cross_tables, ignore_index=True) if cross_tables else pd.DataFrame()
save_csv(personas_by_segment, "personas_by_segment.csv")
plotly_table(personas_by_segment.sort_values("lift_vs_global", ascending=False).head(30))

In [ ]:
# Diagnóstico de concentração por metadados
# Objetivo: avisar quando uma persona parece ser mais um recorte de projeto/marca/público do que um tipo de cliente.
metadata_diag_rows = []
for pid, g in personas_doc_level.groupby("persona_id"):
    row = {"persona_id": pid, "persona_nome_auto": g["persona_nome_auto"].iloc[0], "n_entrevistas": len(g)}
    for c in categorical_cols:
        if c in g.columns:
            vc = g[c].fillna("nao_inferido").astype(str).value_counts()
            row[f"{c}_top_value"] = vc.index[0] if len(vc) else ""
            row[f"{c}_top_share"] = float(vc.iloc[0] / len(g)) if len(vc) else np.nan
    metadata_diag_rows.append(row)

persona_metadata_diagnostics = pd.DataFrame(metadata_diag_rows)

# Flag simples: concentração muito alta em projeto/marca/público não invalida a persona,
# mas indica que vale revisar as evidências manualmente.
flag_cols = [c for c in ["projeto_top_share", "marca_foco_top_share", "publico_top_share", "pais_top_share", "tipo_sessao_top_share"] if c in persona_metadata_diagnostics.columns]
if flag_cols:
    persona_metadata_diagnostics["max_metadata_concentration"] = persona_metadata_diagnostics[flag_cols].max(axis=1)
    persona_metadata_diagnostics["review_flag"] = np.where(
        persona_metadata_diagnostics["max_metadata_concentration"] >= 0.80,
        "revisar: persona muito concentrada em um metadado",
        "ok"
    )
else:
    persona_metadata_diagnostics["max_metadata_concentration"] = np.nan
    persona_metadata_diagnostics["review_flag"] = "sem metadados para avaliar"

save_csv(persona_metadata_diagnostics, "persona_metadata_diagnostics.csv")
plotly_table(persona_metadata_diagnostics.sort_values("max_metadata_concentration", ascending=False))


## 14. Visualizações principais

In [ ]:
# Distribuição de personas — Plotly
plot_df = persona_counts.merge(
    pd.DataFrame({"persona_id": list(persona_name_map.keys()), "persona_nome_auto": list(persona_name_map.values())}),
    on="persona_id", how="left"
)
plot_df["persona_label"] = plot_df["persona_nome_auto"].fillna(plot_df["persona_id"])
plot_df["share_pct"] = plot_df["share_entrevistas"] * 100
plot_df = plot_df.sort_values("n_entrevistas", ascending=True)

fig = px.bar(
    plot_df,
    x="n_entrevistas",
    y="persona_label",
    orientation="h",
    text="n_entrevistas",
    custom_data=["persona_id", "share_pct"],
    title="Tamanho das personas",
    labels={"n_entrevistas": "Número de entrevistas", "persona_label": "Persona"},
)
fig.update_traces(
    textposition="outside",
    hovertemplate="%{customdata[0]}<br>%{y}<br>Entrevistas: %{x}<br>Share: %{customdata[1]:.1f}%<extra></extra>"
)
fig.update_layout(
    template="plotly_white",
    margin=dict(l=220, r=40, t=70, b=50),
    height=max(430, 110 + 75 * len(plot_df)),
    xaxis=dict(range=[0, plot_df["n_entrevistas"].max() * 1.15]),
    showlegend=False,
)
save_plotly_fig(fig, "personas_tamanho_v5_plotly", width=1050, height=max(430, 110 + 75 * len(plot_df)))

In [ ]:
# Heatmap interativo de lifts das dimensões — Plotly
heat = dim_by_persona.pivot_table(index="persona_id", columns="dimension", values="lift_vs_global", fill_value=0)

# Seleciona dimensões mais informativas e preserva labels legíveis
selected_dims = heat.var(axis=0).sort_values(ascending=False).head(14).index.tolist()
heat2 = heat[selected_dims].copy()

# Ordena personas pelo nome manual/auto para facilitar leitura
heat2 = heat2.loc[sorted(heat2.index)]

persona_y_labels = [
    f"{pid}<br><b>{wrap_label(persona_name_map.get(pid, pid), 26)}</b>"
    for pid in heat2.index
]
dim_x_labels = [
    wrap_label(DIMENSION_LABELS.get(d.replace("dim__", "").replace("att__", ""), d), 18)
    for d in selected_dims
]

z = heat2.values
text = np.vectorize(lambda v: f"{v:.1f}x")(z)

fig = go.Figure(data=go.Heatmap(
    z=z,
    x=dim_x_labels,
    y=persona_y_labels,
    text=text,
    texttemplate="%{text}",
    textfont={"size": 12},
    colorbar=dict(title="Lift vs<br>média"),
    hovertemplate="Persona: %{y}<br>Dimensão: %{x}<br>Lift: %{z:.2f}x<extra></extra>",
))
fig.update_layout(
    title="Dimensões que diferenciam cada persona",
    xaxis_title="Dimensão comportamental",
    yaxis_title="Persona",
    template="plotly_white",
    margin=dict(l=210, r=40, t=85, b=150),
    height=max(520, 170 + 95 * heat2.shape[0]),
)
fig.update_xaxes(tickangle=0, automargin=True)
fig.update_yaxes(automargin=True)
save_plotly_fig(fig, "personas_dimensoes_heatmap_v5_plotly", width=1350, height=max(520, 170 + 95 * heat2.shape[0]))

# Também salva uma tabela longa, mais fácil de filtrar/ordenar
heat_long = (
    heat2.reset_index()
    .melt(id_vars="persona_id", var_name="dimension", value_name="lift_vs_global")
)
heat_long["persona_nome"] = heat_long["persona_id"].map(persona_name_map).fillna(heat_long["persona_id"])
heat_long["dimension_label"] = heat_long["dimension"].map(lambda d: DIMENSION_LABELS.get(str(d).replace("dim__", "").replace("att__", ""), d))
heat_long = heat_long.sort_values(["persona_id", "lift_vs_global"], ascending=[True, False])
save_csv(heat_long, "personas_dimensoes_heatmap_long.csv")
plotly_table(
    heat_long[["persona_id", "persona_nome", "dimension_label", "lift_vs_global"]],
    "Dimensões que diferenciam cada persona — tabela longa",
    "tabela_personas_dimensoes_heatmap_long",
    max_rows=80,
)

In [ ]:
# Mapa 2D das entrevistas com SVD do X_model para visualização — Plotly
svd2 = TruncatedSVD(n_components=2, random_state=RANDOM_STATE)
coords = svd2.fit_transform(X_model)
coords_df = personas_doc_level.copy()
coords_df["x"] = coords[:, 0]
coords_df["y"] = coords[:, 1]
coords_df["persona_label"] = coords_df["persona_nome_auto"].fillna(coords_df["persona_id"])
save_csv(coords_df, "personas_doc_coords_2d.csv")

hover_cols = [c for c in ["doc_id", "persona_id", "persona_label", "projeto", "pais", "marca_foco", "publico", "tipo_sessao", "n_words_persona_text"] if c in coords_df.columns]
fig = px.scatter(
    coords_df,
    x="x",
    y="y",
    color="persona_label",
    hover_data=hover_cols,
    title="Mapa 2D das entrevistas por persona",
    labels={"x": "Componente visual 1", "y": "Componente visual 2", "persona_label": "Persona"},
)
fig.update_traces(marker=dict(size=9, opacity=0.78))
fig.update_layout(
    template="plotly_white",
    margin=dict(l=60, r=260, t=80, b=60),
    legend=dict(title="Persona", orientation="v", y=1, x=1.02),
)
save_plotly_fig(fig, "personas_mapa_2d_v5_plotly", width=1150, height=700)

## 15. Relatório Markdown automático

Este relatório é um ponto de partida para apresentação. O mais importante é revisar as evidências e renomear as personas com linguagem de negócio.

In [ ]:
lines = []
lines.append("# Relatório de Personas — entrevistas Kyra")
lines.append("")
lines.append(f"Gerado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
lines.append("")
lines.append("## Leitura executiva")
lines.append("")
lines.append(f"Foram analisadas **{len(persona_docs)} entrevistas** com texto suficiente. A solução final usou **{FINAL_K} personas**.")
lines.append("")
lines.append(f"O maior silhouette puro sugeria k={SILHOUETTE_K}, mas a v2 usa k={BEST_K} como padrão para evitar micro-personas pouco acionáveis e reduzir artefatos de projeto/metadados.")
lines.append("")
lines.append("A recomendação é usar este resultado como uma camada complementar aos clusters de tema: personas explicam padrões de decisão e linguagem do cliente, enquanto tópicos explicam assuntos discutidos.")
lines.append("")

for _, row in persona_summary.iterrows():
    pid = row["persona_id"]
    lines.append(f"## {pid} — {row['persona_nome_auto']}")
    lines.append("")
    lines.append(f"- Tamanho: **{row['n_entrevistas']} entrevistas** ({row['share_entrevistas']:.1%} da base modelada).")
    lines.append(f"- Dimensões dominantes: {row['top_dimensoes']}")
    lines.append(f"- Termos característicos: {row['top_termos']}")
    for c in categorical_cols:
        val = row.get(f"top_{c}", "")
        if isinstance(val, str) and val.strip():
            lines.append(f"- Maior presença em `{c}`: {val}")
    if "persona_metadata_diagnostics" in globals() and not persona_metadata_diagnostics.empty:
        diag = persona_metadata_diagnostics[persona_metadata_diagnostics["persona_id"] == pid]
        if not diag.empty:
            lines.append(f"- Diagnóstico de concentração: {diag.iloc[0].get('review_flag', '')}")
    lines.append("")
    lines.append("**Evidências iniciais**")
    ev = evidence[evidence["persona_id"] == pid].head(5)
    if ev.empty:
        lines.append("- Sem evidências selecionadas automaticamente.")
    else:
        for _, er in ev.iterrows():
            meta = ", ".join([str(er.get(c, "")) for c in ["projeto", "pais", "marca_foco", "publico"] if pd.notna(er.get(c, None)) and str(er.get(c, "")).strip()])
            lines.append(f"- \"{er['excerpt']}\"  ")
            if meta:
                lines.append(f"  - _{meta}_")
    lines.append("")

lines.append("## Próximos ajustes recomendados")
lines.append("")
lines.append("1. Renomear manualmente as personas após ler 5 a 10 evidências de cada uma.")
lines.append("2. Revisar personas marcadas com alta concentração por projeto/marca/público; elas podem ser segmentos reais ou artefatos da amostra.")
lines.append("3. Ajustar os dicionários de dimensões quando alguma palavra estiver poluindo a leitura.")
lines.append("4. Comparar `FINAL_K = 4`, `5` e `6`. Use `SILHOUETTE_K` só se precisar de uma leitura mais granular.")
lines.append("5. Cruzar personas com marca/projeto/público para identificar oportunidades de posicionamento, embalagem, comunicação e canal.")

report_text = "\n".join(lines)
report_path = save_md(report_text, "relatorio_personas.md")
print(report_text[:5000])


## 16. Como interpretar e melhorar

1. Comece pelo `relatorio_personas.md` e pela tabela `personas_summary.csv`.
2. Leia as evidências em `personas_evidence_quotes.csv`; elas são mais importantes do que o nome automático.
3. Abra `persona_metadata_diagnostics.csv` para ver se alguma persona ficou concentrada demais em projeto, marca, país ou público.
4. Compare soluções com `FINAL_K = 4`, `5` e `6`. A v2 evita escolher automaticamente k muito alto só porque o silhouette subiu.
5. Depois de validar qualitativamente, renomeie as personas em linguagem de negócio.

**Regra prática:** uma boa persona deve descrever um padrão de decisão do cliente, não apenas um tema da entrevista nem um projeto específico da base.


## 17. Comparativo de soluções: 3 personas macro vs solução granular

A solução com **3 personas** costuma ser melhor para apresentação executiva, porque força uma leitura simples e acionável. A solução granular, por outro lado, é útil para diagnóstico: ela pode separar subgrupos, tensões específicas e oportunidades mais finas.

Esta seção roda as duas soluções em paralelo e salva outputs separados. A ideia é você poder responder duas perguntas diferentes:

- **Macro:** quais são os grandes tipos de clientes?
- **Granular:** quais subtipos ou nuances aparecem dentro desses grandes grupos?

In [ ]:

# Comparativo v3: solução macro e solução granular
# -------------------------------------------------
# A solução macro fica fixa em 3 personas.
# A granular usa o k de maior silhouette puro, com piso de 5 quando possível.

MACRO_K = 3
if "SILHOUETTE_K" in globals():
    GRANULAR_K = int(max(5, SILHOUETTE_K)) if X_model.shape[0] >= 20 else int(SILHOUETTE_K)
else:
    GRANULAR_K = min(8, max(5, X_model.shape[0] // 8))
GRANULAR_K = min(GRANULAR_K, max(3, X_model.shape[0] - 1))

print(f"Solução macro: k={MACRO_K}")
print(f"Solução granular: k={GRANULAR_K}")

SOLUTIONS_DIR = OUTPUT_DIR / "solutions"
SOLUTIONS_DIR.mkdir(parents=True, exist_ok=True)


def _safe_filename(text):
    return re.sub(r"[^a-zA-Z0-9_\-]+", "_", str(text)).strip("_").lower()



# Nomes manuais para solução macro k=3 gerada na seção comparativa.
# O ID usa o slug da solução, por isso o prefixo muda para macro_k3_pXX.
MANUAL_PERSONA_NAMES_BY_SOLUTION_ID = {
    "macro_k3_p00": "Cuidadora Consciente",
    "macro_k3_p01": "Comparadora Omnicanal",
    "macro_k3_p02": "Compradora Social e Presenteadora",
}

MANUAL_PERSONA_DESCRIPTIONS_BY_SOLUTION_ID = {
    "macro_k3_p00": "Cliente ligada a pele, cuidado, naturalidade, eficácia, sensorialidade, inovação e representatividade.",
    "macro_k3_p01": "Cliente que compara marcas e canais, avalia Natura/Boticário/Avon/loja/consultora, e decide por confiança, facilidade e valor.",
    "macro_k3_p02": "Cliente que compra em contexto social/familiar, presente, indicação, WhatsApp e consultora, mas também verbaliza críticas e objeções.",
}

def infer_persona_names(solution_docs, solution_dim_lifts, solution_terms):
    """Cria nomes iniciais de personas a partir de dimensões com maior lift."""
    name_map = {}
    for pid, g in solution_docs.groupby("persona_id"):
        top_dims = (
            solution_dim_lifts[solution_dim_lifts["persona_id"] == pid]
            .sort_values("lift_vs_global", ascending=False)
            .head(2)
        )
        labels = []
        for d in top_dims["dimension"].tolist():
            key = str(d).replace("dim__", "").replace("att__", "")
            labels.append(DIMENSION_LABELS.get(key, key.replace("_", " ")))
        if labels:
            name = " + ".join(labels)
        else:
            top_terms = solution_terms[solution_terms["persona_id"] == pid]["term"].head(3).astype(str).tolist()
            name = " / ".join(top_terms) if top_terms else pid
        name_map[pid] = name[:160]
    return name_map


def build_solution(k: int, solution_name: str, random_state: int = RANDOM_STATE):
    """Roda uma solução completa de personas para um k específico."""
    solution_slug = _safe_filename(solution_name)
    out_dir = SOLUTIONS_DIR / solution_slug
    tables_dir = out_dir / "tables"
    figures_dir = out_dir / "figures"
    reports_dir = out_dir / "reports"
    for p in [tables_dir, figures_dir, reports_dir]:
        p.mkdir(parents=True, exist_ok=True)

    km = KMeans(n_clusters=k, random_state=random_state, n_init=80)
    labels = km.fit_predict(X_model)

    docs_sol = model_docs.copy()
    docs_sol["solution"] = solution_name
    docs_sol["persona_num"] = labels
    docs_sol["persona_id"] = docs_sol["persona_num"].map(lambda x: f"{solution_slug}_p{int(x):02d}")

    # Junta dimensões interpretáveis
    docs_sol = docs_sol.merge(dim_features, on="doc_id", how="left")
    docs_sol = docs_sol.fillna(0)

    # Lifts de dimensões
    available_dim_cols = [c for c in dim_cols if c in docs_sol.columns]
    dim_mean_global = docs_sol[available_dim_cols].mean().replace(0, np.nan)
    rows = []
    for pid, g in docs_sol.groupby("persona_id"):
        means = g[available_dim_cols].mean()
        lifts = (means / dim_mean_global).replace([np.inf, -np.inf], np.nan).fillna(0)
        for col in available_dim_cols:
            rows.append({
                "solution": solution_name,
                "k": k,
                "persona_id": pid,
                "dimension": col.replace("dim__", "").replace("att__", ""),
                "mean_per_1000_words": means[col],
                "global_mean_per_1000_words": dim_mean_global[col],
                "lift_vs_global": lifts[col],
            })
    dim_lifts = pd.DataFrame(rows).sort_values(["persona_id", "lift_vs_global"], ascending=[True, False])

    # Termos característicos por persona
    persona_text_agg = docs_sol.groupby("persona_id")["persona_text_norm"].apply(lambda s: " ".join(s.dropna())).reset_index()
    cv = CountVectorizer(max_features=6000, min_df=1, ngram_range=(1, 3), stop_words=CUSTOM_STOPWORDS, strip_accents="unicode")
    X_counts = cv.fit_transform(persona_text_agg["persona_text_norm"])
    terms = np.array(cv.get_feature_names_out())
    persona_term_counts = pd.DataFrame(X_counts.toarray(), index=persona_text_agg["persona_id"], columns=terms)
    global_counts = persona_term_counts.sum(axis=0) + 1
    persona_totals = persona_term_counts.sum(axis=1).replace(0, np.nan)
    global_freq = global_counts / global_counts.sum()
    term_rows = []
    for pid in persona_term_counts.index:
        freq = (persona_term_counts.loc[pid] + 1) / (persona_totals.loc[pid] + len(terms))
        lift = freq / global_freq
        score = np.log1p(persona_term_counts.loc[pid]) * np.log1p(lift)
        top = score.sort_values(ascending=False).head(35)
        for term, val in top.items():
            term_rows.append({
                "solution": solution_name,
                "k": k,
                "persona_id": pid,
                "term": term,
                "score": float(val),
                "count": int(persona_term_counts.loc[pid, term]),
                "lift": float(lift[term]),
            })
    terms_sol = pd.DataFrame(term_rows)

    # Nome automático + override manual para a solução macro k=3
    name_auto_map = infer_persona_names(docs_sol, dim_lifts, terms_sol)
    name_map = {pid: MANUAL_PERSONA_NAMES_BY_SOLUTION_ID.get(pid, auto_name) for pid, auto_name in name_auto_map.items()}
    docs_sol["persona_nome_auto_original"] = docs_sol["persona_id"].map(name_auto_map)
    docs_sol["persona_nome_auto"] = docs_sol["persona_id"].map(name_map)
    docs_sol["persona_descricao_manual"] = docs_sol["persona_id"].map(MANUAL_PERSONA_DESCRIPTIONS_BY_SOLUTION_ID).fillna("")
    dim_lifts["persona_nome_auto"] = dim_lifts["persona_id"].map(name_map)
    terms_sol["persona_nome_auto"] = terms_sol["persona_id"].map(name_map)

    # Evidências por persona
    evidence_sol = pd.DataFrame()
    if "chunks" in globals() and len(chunks) and "CHUNK_TEXT_COL" in globals() and CHUNK_TEXT_COL in chunks.columns:
        chunk_evidence_cols = [c for c in [
            "doc_id", "chunk_id", "chunk_index", CHUNK_TEXT_COL, "projeto", "pais", "marca_foco", "publico", "tipo_sessao",
            "cluster_label", "cluster_auto_label", "nmf_topic_label", "lda_topic_label"
        ] if c in chunks.columns]
        chunks_ev = chunks[chunk_evidence_cols].merge(
            docs_sol[["doc_id", "persona_id", "persona_nome_auto"]], on="doc_id", how="inner"
        )
        chunks_ev["text_norm"] = chunks_ev[CHUNK_TEXT_COL].map(norm_text)
        chunks_ev["n_words"] = chunks_ev["text_norm"].str.split().map(len)
        persona_top_dims = (
            dim_lifts.sort_values(["persona_id", "lift_vs_global"], ascending=[True, False])
            .groupby("persona_id")
            .head(4)
            .groupby("persona_id")["dimension"]
            .apply(list)
            .to_dict()
        )
        def _evidence_score(row):
            dims = persona_top_dims.get(row["persona_id"], [])
            score = 0
            for dim in dims:
                pat = compiled_patterns.get(dim)
                if pat:
                    score += count_pattern_hits(row["text_norm"], pat)
            return score + 0.35 * min(row.get("n_words", 0), 120) / 120
        chunks_ev["evidence_score"] = chunks_ev.apply(_evidence_score, axis=1)
        chunks_ev = chunks_ev[chunks_ev["n_words"].between(25, 240, inclusive="both")].copy()
        ex_rows = []
        for pid, g in chunks_ev.sort_values("evidence_score", ascending=False).groupby("persona_id"):
            used_docs = set()
            selected = []
            for _, r in g.iterrows():
                if r["doc_id"] in used_docs and len(used_docs) < min(6, g["doc_id"].nunique()):
                    continue
                selected.append(r)
                used_docs.add(r["doc_id"])
                if len(selected) >= 10:
                    break
            for rank, r in enumerate(selected, 1):
                ex_rows.append({
                    "solution": solution_name,
                    "k": k,
                    "persona_id": pid,
                    "persona_nome_auto": r["persona_nome_auto"],
                    "rank": rank,
                    "doc_id": r["doc_id"],
                    "chunk_id": r.get("chunk_id", None),
                    "evidence_score": r["evidence_score"],
                    "excerpt": short_excerpt(r[CHUNK_TEXT_COL], 620),
                    "projeto": r.get("projeto", None),
                    "pais": r.get("pais", None),
                    "marca_foco": r.get("marca_foco", None),
                    "publico": r.get("publico", None),
                    "tipo_sessao": r.get("tipo_sessao", None),
                })
        evidence_sol = pd.DataFrame(ex_rows)

    # Resumo por persona
    summary_rows = []
    for pid, g in docs_sol.groupby("persona_id"):
        top_dims = dim_lifts[dim_lifts["persona_id"] == pid].head(8)
        top_terms = terms_sol[terms_sol["persona_id"] == pid].head(15)
        ev = evidence_sol[evidence_sol["persona_id"] == pid].head(3) if not evidence_sol.empty else pd.DataFrame()
        row = {
            "solution": solution_name,
            "k": k,
            "persona_id": pid,
            "persona_nome_auto": name_map.get(pid, pid),
            "n_entrevistas": len(g),
            "share_entrevistas": len(g) / len(docs_sol),
            "top_dimensoes": "; ".join([
                f"{DIMENSION_LABELS.get(str(d), str(d).replace('_',' '))} (lift {lift:.1f}x)"
                for d, lift in zip(top_dims["dimension"], top_dims["lift_vs_global"])
            ]),
            "top_termos": ", ".join(top_terms["term"].astype(str).tolist()),
            "evidencias_curtas": " || ".join(ev["excerpt"].astype(str).tolist()) if not ev.empty and "excerpt" in ev.columns else "",
        }
        for c in categorical_cols:
            if c in g.columns:
                vc = g[c].fillna("nao_inferido").astype(str).value_counts().head(3)
                row[f"top_{c}"] = "; ".join([f"{idx}: {val}" for idx, val in vc.items()])
                row[f"{c}_top_share"] = float(vc.iloc[0] / len(g)) if len(vc) else np.nan
        summary_rows.append(row)
    summary_sol = pd.DataFrame(summary_rows).sort_values("n_entrevistas", ascending=False)

    # Salva tabelas
    docs_cols = [c for c in ["doc_id", "solution", "persona_id", "persona_nome_auto", "n_words_persona_text", *categorical_cols] if c in docs_sol.columns]
    docs_sol[docs_cols].to_csv(tables_dir / "personas_doc_level.csv", index=False, encoding="utf-8-sig")
    summary_sol.to_csv(tables_dir / "personas_summary.csv", index=False, encoding="utf-8-sig")
    dim_lifts.to_csv(tables_dir / "personas_dimension_lifts.csv", index=False, encoding="utf-8-sig")
    terms_sol.to_csv(tables_dir / "personas_top_terms.csv", index=False, encoding="utf-8-sig")
    if not evidence_sol.empty:
        evidence_sol.to_csv(tables_dir / "personas_evidence_quotes.csv", index=False, encoding="utf-8-sig")

    # Figura de distribuição
    plt.figure(figsize=(10, 4))
    plot_counts = summary_sol.sort_values("persona_id")
    plt.bar(plot_counts["persona_id"], plot_counts["n_entrevistas"])
    plt.title(f"Distribuição de personas — {solution_name}")
    plt.xlabel("Persona")
    plt.ylabel("Nº de entrevistas")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(figures_dir / "personas_distribution.png", dpi=180)
    plt.show()

    return {
        "name": solution_name,
        "slug": solution_slug,
        "k": k,
        "model": km,
        "docs": docs_sol,
        "summary": summary_sol,
        "dim_lifts": dim_lifts,
        "terms": terms_sol,
        "evidence": evidence_sol,
        "name_map": name_map,
        "out_dir": out_dir,
        "tables_dir": tables_dir,
        "figures_dir": figures_dir,
        "reports_dir": reports_dir,
    }

solutions = {}
solutions["macro_k3"] = build_solution(MACRO_K, "macro_k3")
solutions["granular"] = build_solution(GRANULAR_K, f"granular_k{GRANULAR_K}")

# Tabela comparativa consolidada
comparison_summary = pd.concat([s["summary"] for s in solutions.values()], ignore_index=True)
comparison_summary.to_csv(OUTPUT_DIR / "tables" / "personas_comparativo_macro_granular.csv", index=False, encoding="utf-8-sig")
plotly_table(comparison_summary[["solution", "persona_id", "persona_nome_auto", "n_entrevistas", "share_entrevistas", "top_dimensoes"]])



## 18. Análise de linguagem por persona

Esta seção transforma os termos característicos em uma leitura mais útil para comunicação: quais palavras aparecem, que tipo de linguagem a persona usa e quais mensagens podem ressoar melhor.

Ela não substitui leitura qualitativa, mas ajuda a encontrar frases, argumentos e gatilhos que podem ir para roteiro, posicionamento, copy, naming, embalagem ou material de vendas.

In [ ]:

# Análise de linguagem por persona
# --------------------------------
# Gera uma tabela para cada solução com:
# - vocabulário característico;
# - linguagem sugerida;
# - possíveis mensagens;
# - palavras/temas a observar manualmente.

LANGUAGE_CUE_RULES = [
    ("preco_valor_promocao", "linguagem objetiva de valor, economia, vantagem e comparação de preço"),
    ("eficacia_resultado_funcional", "linguagem de prova, resultado, benefício funcional e desempenho"),
    ("sensorial_perfume_prazer", "linguagem sensorial, emocional, de prazer de uso, cheiro, textura e ritual"),
    ("natural_sustentavel_consciente", "linguagem consciente, natural, sustentável, ingredientes e impacto"),
    ("marca_confianca_tradicao", "linguagem de confiança, tradição, segurança e credibilidade da marca"),
    ("status_presente_estetica", "linguagem estética, presenteável, premium, bonita e aspiracional"),
    ("praticidade_rotina_conveniencia", "linguagem de rotina, facilidade, tempo, praticidade e conveniência"),
    ("canal_loja_consultora_venda", "linguagem de canal, atendimento, loja, consultora, revista, WhatsApp e compra"),
    ("pele_corpo_cuidado_pessoal", "linguagem de autocuidado, pele/corpo, cuidado pessoal e necessidade específica"),
    ("familia_social_influencia", "linguagem social, familiar, indicação, rede de confiança e recomendação"),
    ("diversidade_identidade_representatividade", "linguagem de identidade, representatividade, autoestima e inclusão"),
    ("inovacao_tecnologia_novidade", "linguagem de novidade, tecnologia, modernidade, curiosidade e experimentação"),
    ("negativo_rejeicao", "atenção a vocabulário de rejeição, fricção e dor"),
    ("duvida_ceticismo", "atenção a linguagem de dúvida, indecisão, necessidade de explicação e prova"),
    ("comparacao_concorrencia", "linguagem comparativa: cliente avalia alternativas e marcas concorrentes"),
]


def language_cues_for_dims(top_dim_keys):
    cues = []
    for key, cue in LANGUAGE_CUE_RULES:
        if key in top_dim_keys:
            cues.append(cue)
    return cues


def build_language_profile(solution):
    rows = []
    summary = solution["summary"]
    dim_lifts = solution["dim_lifts"]
    terms = solution["terms"]
    for _, srow in summary.iterrows():
        pid = srow["persona_id"]
        top_dims_df = dim_lifts[dim_lifts["persona_id"] == pid].sort_values("lift_vs_global", ascending=False).head(6)
        top_dim_keys = top_dims_df["dimension"].astype(str).tolist()
        top_terms = terms[terms["persona_id"] == pid].sort_values("score", ascending=False).head(25)["term"].astype(str).tolist()
        cues = language_cues_for_dims(top_dim_keys)
        rows.append({
            "solution": solution["name"],
            "persona_id": pid,
            "persona_nome_auto": srow["persona_nome_auto"],
            "n_entrevistas": srow["n_entrevistas"],
            "top_dimensoes_linguagem": "; ".join([DIMENSION_LABELS.get(k, k.replace("_", " ")) for k in top_dim_keys]),
            "vocabulario_caracteristico": ", ".join(top_terms[:18]),
            "como_essa_persona_fala": "; ".join(cues[:5]),
            "mensagem_que_tende_a_funcionar": " | ".join([
                "Mostre benefício concreto e fácil de entender" if "eficacia_resultado_funcional" in top_dim_keys else "",
                "Explicite custo-benefício, promoção ou vantagem" if "preco_valor_promocao" in top_dim_keys else "",
                "Use linguagem sensorial e de experiência" if "sensorial_perfume_prazer" in top_dim_keys else "",
                "Traga ingredientes, naturalidade e impacto" if "natural_sustentavel_consciente" in top_dim_keys else "",
                "Use prova social, indicação e confiança" if "familia_social_influencia" in top_dim_keys or "marca_confianca_tradicao" in top_dim_keys else "",
                "Reduza incerteza com comparação, demonstração e explicação" if "duvida_ceticismo" in top_dim_keys else "",
            ]).replace(" |  | ", " | ").strip(" |"),
            "alerta_de_linguagem": "Revisar falas negativas/objeções" if "negativo_rejeicao" in top_dim_keys else ("Persona pode precisar de mais prova/clareza" if "duvida_ceticismo" in top_dim_keys else ""),
        })
    return pd.DataFrame(rows)

language_profiles = []
for solution in solutions.values():
    lang = build_language_profile(solution)
    lang.to_csv(solution["tables_dir"] / "persona_language_profile.csv", index=False, encoding="utf-8-sig")
    language_profiles.append(lang)

persona_language_profile = pd.concat(language_profiles, ignore_index=True)
persona_language_profile.to_csv(OUTPUT_DIR / "tables" / "persona_language_profile_comparativo.csv", index=False, encoding="utf-8-sig")
plotly_table(persona_language_profile)


## 19. Oportunidades por persona

Aqui o notebook converte a leitura das personas em hipóteses acionáveis. O objetivo é criar uma ponte entre análise qualitativa e decisões de negócio:

- oportunidade de produto;
- oportunidade de comunicação;
- oportunidade de canal;
- oportunidade de preço/promoção;
- principais riscos ou barreiras.

Essas recomendações são geradas por regras simples em cima das dimensões dominantes. Use como ponto de partida e revise manualmente com as evidências.

In [ ]:

# Oportunidades por persona
# -------------------------

def opportunity_rules(top_dim_keys):
    produto, comunicacao, canal, preco, riscos = [], [], [], [], []

    if "preco_valor_promocao" in top_dim_keys:
        preco.append("Testar bundles, kits de entrada, descontos claros e comunicação de custo-benefício.")
        comunicacao.append("Explicitar por que o produto vale o preço, evitando só linguagem aspiracional.")
        riscos.append("Pode rejeitar proposta se o preço parecer desconectado do benefício percebido.")
    if "eficacia_resultado_funcional" in top_dim_keys:
        produto.append("Priorizar claims funcionais, antes/depois, duração, performance e benefício mensurável.")
        comunicacao.append("Usar prova, demonstração e linguagem concreta de resultado.")
        riscos.append("Promessas vagas podem gerar ceticismo ou frustração.")
    if "sensorial_perfume_prazer" in top_dim_keys:
        produto.append("Explorar fragrância, textura, toque, ritual de uso e experiência sensorial.")
        comunicacao.append("Usar descrições sensoriais e emocionais, não só funcionais.")
    if "natural_sustentavel_consciente" in top_dim_keys:
        produto.append("Destacar ingredientes, refil, impacto ambiental, naturalidade e rastreabilidade.")
        comunicacao.append("Mostrar sustentabilidade de forma simples e verificável.")
        riscos.append("Evitar greenwashing ou claims genéricos sem prova.")
    if "marca_confianca_tradicao" in top_dim_keys:
        comunicacao.append("Ancorar a proposta em confiança, tradição, reputação e consistência da marca.")
        riscos.append("Mudanças muito bruscas de identidade podem gerar estranhamento.")
    if "status_presente_estetica" in top_dim_keys:
        produto.append("Trabalhar embalagem, kits presenteáveis, edição especial e aparência premium.")
        comunicacao.append("Valorizar estética, desejo, ocasião de presente e orgulho de entregar/usar.")
    if "praticidade_rotina_conveniencia" in top_dim_keys:
        produto.append("Simplificar modo de uso, tamanho, aplicação, refill e integração na rotina.")
        comunicacao.append("Mostrar como entra no dia a dia sem esforço.")
        canal.append("Priorizar experiência de compra rápida, reposição e lembretes.")
    if "canal_loja_consultora_venda" in top_dim_keys:
        canal.append("Fortalecer loja/consultora/WhatsApp com material comparativo, argumentos e amostras.")
        comunicacao.append("Criar scripts de venda por persona e objeção.")
    if "pele_corpo_cuidado_pessoal" in top_dim_keys:
        produto.append("Segmentar por necessidade específica de pele/corpo/cabelo e momento de cuidado.")
        comunicacao.append("Conectar benefício técnico com autocuidado e sensação na pele.")
    if "familia_social_influencia" in top_dim_keys:
        canal.append("Explorar indicação, prova social, kits para família/presente e compartilhamento.")
        comunicacao.append("Usar linguagem de recomendação, confiança e experiências reais.")
    if "diversidade_identidade_representatividade" in top_dim_keys:
        produto.append("Garantir adequação para diferentes tons, tipos de pele/cabelo e necessidades reais.")
        comunicacao.append("Trazer representatividade autêntica e linguagem inclusiva.")
        riscos.append("A persona tende a perceber rapidamente comunicação pouco autêntica.")
    if "inovacao_tecnologia_novidade" in top_dim_keys:
        produto.append("Testar novidades, tecnologia, claims modernos e formatos diferentes.")
        comunicacao.append("Explicar a inovação de forma tangível, sem jargão excessivo.")
    if "negativo_rejeicao" in top_dim_keys:
        riscos.append("Há sinais de rejeição explícita; priorizar leitura das evidências antes de tomar decisão.")
        comunicacao.append("Atacar objeções diretamente, com prova e linguagem transparente.")
    if "duvida_ceticismo" in top_dim_keys:
        riscos.append("Indecisão/ceticismo: precisa de comparação, teste, amostra ou explicação simples.")
        canal.append("Amostras, consultoria assistida e demonstração podem ajudar conversão.")
    if "comparacao_concorrencia" in top_dim_keys:
        comunicacao.append("Mostrar diferenciais frente a alternativas conhecidas.")
        riscos.append("Comparação com concorrentes é parte central da decisão.")

    def uniq_join(items):
        out = []
        for item in items:
            if item and item not in out:
                out.append(item)
        return " ".join(out[:4])

    return {
        "oportunidade_produto": uniq_join(produto),
        "oportunidade_comunicacao": uniq_join(comunicacao),
        "oportunidade_canal": uniq_join(canal),
        "oportunidade_preco_promocao": uniq_join(preco),
        "riscos_barreiras": uniq_join(riscos),
    }


def build_opportunities(solution):
    rows = []
    for _, srow in solution["summary"].iterrows():
        pid = srow["persona_id"]
        top_dims = (
            solution["dim_lifts"][solution["dim_lifts"]["persona_id"] == pid]
            .sort_values("lift_vs_global", ascending=False)
            .head(7)
        )
        top_dim_keys = top_dims["dimension"].astype(str).tolist()
        opp = opportunity_rules(top_dim_keys)
        rows.append({
            "solution": solution["name"],
            "persona_id": pid,
            "persona_nome_auto": srow["persona_nome_auto"],
            "n_entrevistas": srow["n_entrevistas"],
            "share_entrevistas": srow["share_entrevistas"],
            "dimensoes_base": "; ".join([DIMENSION_LABELS.get(k, k.replace("_", " ")) for k in top_dim_keys]),
            **opp,
        })
    return pd.DataFrame(rows)

opportunity_tables = []
for solution in solutions.values():
    opp = build_opportunities(solution)
    opp.to_csv(solution["tables_dir"] / "persona_opportunities.csv", index=False, encoding="utf-8-sig")
    opportunity_tables.append(opp)

persona_opportunities = pd.concat(opportunity_tables, ignore_index=True)
persona_opportunities.to_csv(OUTPUT_DIR / "tables" / "persona_opportunities_comparativo.csv", index=False, encoding="utf-8-sig")
plotly_table(persona_opportunities)


## 20. Classificador de nova entrevista

Esta é a parte voltada para o uso futuro do projeto. Você cola uma entrevista nova no campo `NEW_INTERVIEW_TEXT` e o notebook devolve uma distribuição percentual de aderência às personas.

Exemplo do tipo de leitura esperada:

> 52% Persona A — indecisa/cética  
> 31% Persona B — sensível a preço  
> 17% Persona C — sensorial/experiência

Importante: isso não é uma probabilidade estatística perfeita. É uma **similaridade com os centroides** das personas treinadas. Funciona melhor como triagem e leitura inicial, e deve ser validado lendo os trechos da entrevista.

In [ ]:

# Classificador de nova entrevista
# --------------------------------
# Cole uma entrevista no NEW_INTERVIEW_TEXT e rode a célula.

NEW_INTERVIEW_TEXT = """
Cole aqui o texto de uma nova entrevista.
Pode ser uma transcrição inteira, um resumo longo ou vários trechos juntos.
Quanto mais texto, melhor a classificação.
""".strip()


def vectorize_new_interview(text: str):
    """Transforma uma entrevista nova para a mesma matriz usada no treinamento."""
    text_norm = norm_text(text)
    n_words = max(len(text_norm.split()), 1)

    # Texto
    X_tfidf_new = vectorizer.transform([text_norm])
    X_text_new = svd.transform(X_tfidf_new)
    X_text_scaled_new = text_scaler.transform(X_text_new)

    # Dimensões interpretáveis
    dim_values = {}
    for dim, pat in compiled_patterns.items():
        hits = count_pattern_hits(text_norm, pat)
        dim_values[f"dim__{dim}"] = hits / n_words * 1000
    dim_row = pd.DataFrame([{c: dim_values.get(c, 0.0) for c in dim_feature_cols}]) if dim_feature_cols else pd.DataFrame(index=[0])
    if dim_feature_cols:
        X_dim_new = dim_row[dim_feature_cols].replace([np.inf, -np.inf], 0).fillna(0).to_numpy(dtype=float)
        X_dim_scaled_new = dim_scaler.transform(X_dim_new) if dim_scaler is not None else X_dim_new
    else:
        X_dim_scaled_new = np.empty((1, 0))

    # Auxiliares antigos: para entrevista nova, por padrão ficam zerados.
    # Isso evita depender de cluster/topic antigo para classificar uma entrada nova.
    if aux_feature_cols:
        X_aux_new = np.zeros((1, len(aux_feature_cols)))
        X_aux_scaled_new = aux_scaler.transform(X_aux_new) if aux_scaler is not None else X_aux_new
    else:
        X_aux_scaled_new = np.empty((1, 0))

    X_new = np.hstack([
        X_text_scaled_new * TEXT_WEIGHT,
        X_dim_scaled_new * DIMENSION_WEIGHT,
        X_aux_scaled_new * AUX_OLD_CLUSTER_WEIGHT,
    ])
    X_new = normalize(X_new)

    diagnostics = {
        "n_words": n_words,
        "dim_values": dim_values,
        "text_norm": text_norm,
    }
    return X_new, diagnostics


def classify_new_interview(text: str, solution_key: str = "macro_k3", temperature_scale: float = 0.45, top_n: int | None = None):
    """Classifica uma entrevista nova contra uma solução treinada."""
    if solution_key not in solutions:
        raise ValueError(f"solution_key inválida. Use uma destas: {list(solutions.keys())}")
    solution = solutions[solution_key]
    km = solution["model"]
    X_new, diagnostics = vectorize_new_interview(text)

    distances = km.transform(X_new).ravel()
    # Converte distância em porcentagem de aderência.
    # Distâncias menores = maior aderência.
    positive_dist = distances[np.isfinite(distances)]
    temp = np.median(positive_dist) * temperature_scale if len(positive_dist) else 1.0
    temp = max(float(temp), 1e-6)
    logits = -distances / temp
    logits = logits - logits.max()
    probs = np.exp(logits)
    probs = probs / probs.sum()

    rows = []
    for cluster_num, prob in enumerate(probs):
        persona_id = f"{solution['slug']}_p{cluster_num:02d}"
        rows.append({
            "solution": solution["name"],
            "persona_id": persona_id,
            "persona_nome_auto": solution["name_map"].get(persona_id, persona_id),
            "aderencia_pct": float(prob * 100),
            "distancia_centroide": float(distances[cluster_num]),
        })
    out = pd.DataFrame(rows).sort_values("aderencia_pct", ascending=False)
    if top_n is not None:
        out = out.head(top_n)

    # Diagnóstico das dimensões mais ativadas na entrevista nova
    dim_diag = pd.DataFrame([
        {"dimension": k.replace("dim__", ""), "valor_por_1000_palavras": v}
        for k, v in diagnostics["dim_values"].items()
    ]).sort_values("valor_por_1000_palavras", ascending=False)
    dim_diag["label"] = dim_diag["dimension"].map(lambda x: DIMENSION_LABELS.get(x, x.replace("_", " ")))

    return out, dim_diag, diagnostics

# Exemplo de uso:
# 1) Cole uma entrevista de verdade em NEW_INTERVIEW_TEXT.
# 2) Rode esta célula.
# 3) Veja a classificação macro e granular.

if NEW_INTERVIEW_TEXT and "Cole aqui" not in NEW_INTERVIEW_TEXT:
    macro_result, macro_dims, macro_diag = classify_new_interview(NEW_INTERVIEW_TEXT, "macro_k3")
    granular_result, granular_dims, granular_diag = classify_new_interview(NEW_INTERVIEW_TEXT, "granular")

    print("Classificação macro — 3 personas")
    plotly_table(macro_result)
    print("\nClassificação granular")
    plotly_table(granular_result)
    print("\nDimensões mais ativadas na entrevista nova")
    plotly_table(macro_dims.head(12))
else:
    print("Cole uma entrevista real em NEW_INTERVIEW_TEXT para gerar a classificação.")


## 21. Relatório comparativo final — macro, granular, linguagem e oportunidades

Esta última seção salva um relatório em Markdown juntando as duas soluções, os perfis de linguagem e as oportunidades por persona.

In [ ]:

# Relatório comparativo v3
report_lines = []
report_lines.append("# Relatório comparativo de personas — v4")
report_lines.append("")
report_lines.append(f"Gerado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append("")
report_lines.append("## Como usar")
report_lines.append("")
report_lines.append("- Use `macro_k3` para narrativa executiva e classificação inicial de novas entrevistas. Nesta v4, as 3 personas macro têm nomes manuais: Cuidadora Consciente, Comparadora Omnicanal e Compradora Social e Presenteadora.")
report_lines.append(f"- Use `granular_k{GRANULAR_K}` para diagnóstico fino, identificação de subpersonas e oportunidades específicas.")
report_lines.append("- Para uma entrevista nova, use a função `classify_new_interview(texto, solution_key='macro_k3')`.")
report_lines.append("")

for key, sol in solutions.items():
    report_lines.append(f"## Solução: {sol['name']} — k={sol['k']}")
    report_lines.append("")
    for _, row in sol["summary"].sort_values("n_entrevistas", ascending=False).iterrows():
        pid = row["persona_id"]
        report_lines.append(f"### {row['persona_nome_auto']}")
        report_lines.append("")
        report_lines.append(f"- ID: `{pid}`")
        desc = str(row.get("persona_descricao_manual", "")).strip()
        if desc:
            report_lines.append(f"- Descrição executiva: {desc}")
        report_lines.append(f"- Tamanho: **{int(row['n_entrevistas'])} entrevistas** ({row['share_entrevistas']:.1%}).")
        report_lines.append(f"- Dimensões dominantes: {row['top_dimensoes']}")
        report_lines.append(f"- Termos característicos: {row['top_termos']}")

        lang = persona_language_profile[persona_language_profile["persona_id"] == pid]
        if not lang.empty:
            lrow = lang.iloc[0]
            report_lines.append(f"- Linguagem: {lrow['como_essa_persona_fala']}")
            if str(lrow.get("mensagem_que_tende_a_funcionar", "")).strip():
                report_lines.append(f"- Mensagem que tende a funcionar: {lrow['mensagem_que_tende_a_funcionar']}")

        opp = persona_opportunities[persona_opportunities["persona_id"] == pid]
        if not opp.empty:
            orow = opp.iloc[0]
            for col, label in [
                ("oportunidade_produto", "Produto"),
                ("oportunidade_comunicacao", "Comunicação"),
                ("oportunidade_canal", "Canal"),
                ("oportunidade_preco_promocao", "Preço/promoção"),
                ("riscos_barreiras", "Riscos/barreiras"),
            ]:
                val = str(orow.get(col, "")).strip()
                if val:
                    report_lines.append(f"- {label}: {val}")

        ev = sol["evidence"]
        if ev is not None and not ev.empty:
            ev_pid = ev[ev["persona_id"] == pid].head(2)
            if not ev_pid.empty:
                report_lines.append("- Evidências iniciais:")
                for _, erow in ev_pid.iterrows():
                    report_lines.append(f"  - \"{erow['excerpt']}\"")
        report_lines.append("")

report_path = OUTPUT_DIR / "reports" / "relatorio_personas_comparativo_v4.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print("Relatório salvo em:", report_path)
print("Tabelas comparativas principais:")
print("-", OUTPUT_DIR / "tables" / "personas_comparativo_macro_granular.csv")
print("-", OUTPUT_DIR / "tables" / "persona_language_profile_comparativo.csv")
print("-", OUTPUT_DIR / "tables" / "persona_opportunities_comparativo.csv")

